In [4]:
!pip install mysql-connector-python

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import mysql.connector

In [3]:
mydb=mysql.connector.connect(
    host="localhost",
    user="root",
    password="27665431shanV#",
    port=3306
)

In [4]:
mycursor=mydb.cursor(buffered=True)

In [15]:
mycursor.execute("SHOW DATABASES")
for x in mycursor:
    print(x)

('information_schema',)
('mysql',)
('performance_schema',)
('project',)
('sys',)
('uber_db',)


In [5]:
mycursor.execute("USE uber_db")

In [12]:
!pip install tabulate

Defaulting to user installation because normal site-packages is not writeable


In [6]:
from tabulate import tabulate

In [27]:
mycursor.execute("RENAME TABLE cleaned_uber_eats TO uber_data")

ProgrammingError: 1050 (42S01): Table 'uber_data' already exists

In [5]:
from tabulate import tabulate

mycursor.execute("SELECT * FROM  uber_data")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+------------------------------------------------------+----------------+--------------+--------+---------+-----------------------+-------------------------------+----------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------+-------------------------------+--------------------+-------------------+
| name                                                 | online_order   | book_table   |   rate |   votes | location              | rest_type                     | dish_liked                                                                                                                             | cuisines                                                                             |   approx_cost(for two people) | listed_in(type)    | listed_in(city)   |
|------------------------------------------------------+----------------+-----

In [8]:
mycursor.execute("DESCRIBE uber_data")
out = mycursor.fetchall()

for row in out:
    print(row)

('name', 'text', 'YES', '', None, '')
('online_order', 'text', 'YES', '', None, '')
('book_table', 'text', 'YES', '', None, '')
('rate', 'double', 'YES', '', None, '')
('votes', 'int', 'YES', '', None, '')
('location', 'text', 'YES', '', None, '')
('rest_type', 'text', 'YES', '', None, '')
('dish_liked', 'text', 'YES', '', None, '')
('cuisines', 'text', 'YES', '', None, '')
('approx_cost(for two people)', 'int', 'YES', '', None, '')
('listed_in(type)', 'text', 'YES', '', None, '')
('listed_in(city)', 'text', 'YES', '', None, '')


In [7]:
# Which Bangalore locations have the highest average restaurant ratings?
# Business Value: Identifies premium-performing areas suitable for brand positioning and new partner onboarding.#

mycursor.execute("""
SELECT location, ROUND(AVG(rate),2) AS avg_rating
FROM uber_data
GROUP BY location
ORDER BY avg_rating DESC
LIMIT 1
""")

out = mycursor.fetchall()

print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+---------------+--------------+
| location      |   avg_rating |
|---------------+--------------|
| Church Street |         4.21 |
+---------------+--------------+


In [13]:
# Which locations are over-saturated with restaurants?
# Business Value: Helps avoid overcrowded markets and guides smarter expansion decisions.

mycursor.execute("""
SELECT location
FROM uber_data
GROUP BY location
ORDER BY COUNT(*) DESC
LIMIT 1
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+------------+
| location   |
|------------|
| Jayanagar  |
+------------+


In [11]:
# 3Does online ordering improve restaurant ratings?
 Business Value: Evaluates the ROI of Uber Eats’ online ordering feature for partners.#
mycursor.execute("""
SELECT online_order, 
ROUND(AVG(rate),2) AS avg_rating,
SUM(votes) AS total_votes
FROM uber_data
GROUP BY online_order
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+----------------+--------------+---------------+
| online_order   |   avg_rating |   total_votes |
|----------------+--------------+---------------|
| Yes            |         3.84 |        837163 |
| No             |         3.78 |        223335 |
+----------------+--------------+---------------+


In [29]:
# 4 How do low, mid, and premium-priced restaurants perform in terms of ratings?
 #Business Value: Supports pricing-based market segmentation strategies.
mycursor.execute("""
SELECT 
CASE 
    WHEN `approx_cost(for two people)` < 500 THEN 'Low'
    WHEN `approx_cost(for two people)` <= 1500 THEN 'Mid'
    ELSE 'Premium'
END AS price_range,
ROUND(AVG(rate),2) AS avg_rating,
COUNT(*) AS total_restaurants

FROM uber_data
GROUP BY price_range
ORDER BY avg_rating DESC
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+---------------+--------------+---------------------+
| price_range   |   avg_rating |   total_restaurants |
|---------------+--------------+---------------------|
| Premium       |         4.23 |                  71 |
| Mid           |         3.84 |                1258 |
| Low           |         3.77 |                 815 |
+---------------+--------------+---------------------+


In [14]:
# 5 Which cuisines are most common in Bangalore?
# Business Value: Reveals market demand and cuisine saturation levels.
mycursor.execute("""
SELECT cuisines, COUNT(*) AS total_restaurants
FROM uber_data
GROUP BY cuisines
ORDER BY total_restaurants DESC
LIMIT 1
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-----------------------+---------------------+
| cuisines              |   total_restaurants |
|-----------------------+---------------------|
| North Indian, Chinese |                 115 |
+-----------------------+---------------------+


In [15]:
# 6Which cuisines receive the highest average ratings?
#Business Value: Identifies high-quality cuisine categories suitable for promotion.

mycursor.execute("""
SELECT cuisines,
ROUND(AVG(rate),2) AS avg_rating,
COUNT(*) AS total_restaurants
FROM uber_data
GROUP BY cuisines
HAVING COUNT(*) > 10
ORDER BY avg_rating DESC
LIMIT 1
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+---------------------+--------------+---------------------+
| cuisines            |   avg_rating |   total_restaurants |
|---------------------+--------------+---------------------|
| Ice Cream, Desserts |         4.14 |                  25 |
+---------------------+--------------+---------------------+


In [16]:
#Which cuisines perform well despite having fewer restaurants?
 #Business Value: Highlights niche opportunities for differentiation
mycursor.execute("""
SELECT cuisines,
COUNT(*) AS total_restaurants,
ROUND(AVG(rate),2) AS avg_rating
FROM uber_data
GROUP BY cuisines
HAVING COUNT(*) < 50
ORDER BY avg_rating DESC
LIMIT 1
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+---------------------------------------------------------------+---------------------+--------------+
| cuisines                                                      |   total_restaurants |   avg_rating |
|---------------------------------------------------------------+---------------------+--------------|
| Continental, North Indian, Italian, South Indian, Finger Food |                   3 |          4.9 |
+---------------------------------------------------------------+---------------------+--------------+


In [17]:
#7 Which locations are ideal for premium restaurant onboarding?
 #Business Value: Combines cost, rating, and location insights to guide premium expansion.

mycursor.execute("""
SELECT location,
ROUND(AVG(rate),2) AS avg_rating,
ROUND(AVG(`approx_cost(for two people)`),0) AS avg_cost,
COUNT(*) AS total_restaurants
FROM uber_data
GROUP BY location
HAVING AVG(`approx_cost(for two people)`) > 1000
AND AVG(rate) > 4
ORDER BY avg_rating DESC
LIMIT 1
""")

out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+--------------+--------------+------------+---------------------+
| location     |   avg_rating |   avg_cost |   total_restaurants |
|--------------+--------------+------------+---------------------|
| Lavelle Road |         4.16 |       1318 |                  11 |
+--------------+--------------+------------+---------------------+


In [18]:
#8Which locations show high demand but lower average ratings?
 #Business Value: Indicates areas where quality improvement initiatives are needed.
mycursor.execute("""
SELECT 
    location,
    COUNT(*) AS total_restaurants,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(`approx_cost(for two people)`), 0) AS avg_cost
FROM uber_data
GROUP BY location
HAVING 
    COUNT(*) >= 10
    AND AVG(rate) < 3.8
ORDER BY total_restaurants DESC, avg_rating ASC
LIMIT 1
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+-------------------+---------------------+--------------+------------+
| location          |   total_restaurants |   avg_rating |   avg_cost |
|-------------------+---------------------+--------------+------------|
| Bannerghatta Road |                 233 |         3.63 |        552 |
+-------------------+---------------------+--------------+------------+


In [19]:
#9What combination of factors maximizes restaurant success on Uber Eats?
# (Pricing + Location + Cuisine + Platform Features)
 #Business Value: Supports strategic partner recommendations.
mycursor.execute("""
SELECT 
    location,
    cuisines,
    ROUND(AVG(`approx_cost(for two people)`), 0) AS avg_cost,
    ROUND(AVG(rate), 2) AS avg_rating,
    COUNT(*) AS total_restaurants
FROM uber_data
GROUP BY location, cuisines
HAVING 
    AVG(rate) >= 4.0
    AND COUNT(*) >= 5
ORDER BY avg_rating DESC, total_restaurants DESC, avg_cost DESC
LIMIT 1
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+------------+---------------------------------------------+------------+--------------+---------------------+
| location   | cuisines                                    |   avg_cost |   avg_rating |   total_restaurants |
|------------+---------------------------------------------+------------+--------------+---------------------|
| Bellandur  | North Indian, Chinese, Continental, Mexican |       1400 |          4.4 |                   5 |
+------------+---------------------------------------------+------------+--------------+---------------------+


In [9]:
#10 Do restaurants offering both online ordering and table booking perform better?
 #Business Value: Validates bundled feature adoption for partners.

mycursor.execute("""
SELECT 
    online_order,
    book_table,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(`approx_cost(for two people)`), 0) AS avg_cost,
    COUNT(*) AS total_restaurants
FROM uber_data
GROUP BY online_order, book_table
ORDER BY avg_rating DESC;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))

+----------------+--------------+--------------+------------+---------------------+
| online_order   | book_table   |   avg_rating |   avg_cost |   total_restaurants |
|----------------+--------------+--------------+------------+---------------------|
| Yes            | Yes          |         4.1  |       1049 |                 314 |
| No             | Yes          |         4.1  |       1392 |                 117 |
| Yes            | No           |         3.77 |        506 |                1318 |
| No             | No           |         3.69 |        561 |                 395 |
+----------------+--------------+--------------+------------+---------------------+


In [10]:

mycursor.execute("""
SELECT 
    online_order,
    book_table,
    ROUND(AVG(rate), 2) AS avg_rating,
    ROUND(AVG(`approx_cost(for two people)`), 0) AS avg_cost,
    COUNT(*) AS total_restaurants
FROM uber_data
GROUP BY online_order, book_table
ORDER BY avg_rating DESC;
""")
out = mycursor.fetchall()
print(tabulate(out, headers=[i[0] for i in mycursor.description], tablefmt='psql'))


+----------------+--------------+--------------+------------+---------------------+
| online_order   | book_table   |   avg_rating |   avg_cost |   total_restaurants |
|----------------+--------------+--------------+------------+---------------------|
| Yes            | Yes          |         4.1  |       1049 |                 314 |
| No             | Yes          |         4.1  |       1392 |                 117 |
| Yes            | No           |         3.77 |        506 |                1318 |
| No             | No           |         3.69 |        561 |                 395 |
+----------------+--------------+--------------+------------+---------------------+


In [12]:
mycursor.execute("SHOW TABLES;")
print(mycursor.fetchall())

[('clean_orders (1)',), ('clean_ordersa',), ('cleaned_uber_eats (5)',), ('uber_data',)]


In [14]:
#which one top restaurent.
mycursor.execute("""
SELECT restaurant_name, COUNT(*) AS total_orders
FROM clean_ordersa
GROUP BY restaurant_name
ORDER BY total_orders DESC
LIMIT 10
""")

out = mycursor.fetchall()

print(tabulate(
    out,
    headers=[i[0] for i in mycursor.description],
    tablefmt='psql'
))

+-----------------------------------------+----------------+
| restaurant_name                         |   total_orders |
|-----------------------------------------+----------------|
| Andhra Grills                           |              8 |
| Mum?s Kitchen                           |              7 |
| Carnatic                                |              6 |
| Five Star Chicken                       |              6 |
| Mayura 1989                             |              6 |
| Suryawanshi                             |              6 |
| Tiffin Room - Miraya Hotel & Residences |              6 |
| The Fisherman's Wharf                   |              6 |
| Tandoor Garden                          |              6 |
| Patisserie Nitash                       |              6 |
+-----------------------------------------+----------------+


In [15]:
#Payment Method Analysis
mycursor.execute("""
SELECT payment_method, COUNT(*) AS usage_count
FROM clean_ordersa
GROUP BY payment_method
ORDER BY usage_count DESC
""")

out = mycursor.fetchall()

print(tabulate(
    out,
    headers=[i[0] for i in mycursor.description],
    tablefmt='psql'
))

+------------------+---------------+
| payment_method   |   usage_count |
|------------------+---------------|
| Cash             |          1552 |
| Upi              |          1510 |
| Card             |          1500 |
+------------------+---------------+


In [16]:
#highest rev in restaurant
mycursor.execute("""
SELECT restaurant_name, SUM(order_value) AS total_revenue
FROM clean_ordersa
GROUP BY restaurant_name
ORDER BY total_revenue DESC
LIMIT 10
""")

out = mycursor.fetchall()

print(tabulate(
    out,
    headers=[i[0] for i in mycursor.description],
    tablefmt='psql'
))

+-----------------------+-----------------+
| restaurant_name       |   total_revenue |
|-----------------------+-----------------|
| Andhra Grills         |         9321.05 |
| Patisserie Nitash     |         7313.59 |
| Mayura 1989           |         6911.53 |
| The Chervil           |         6615.55 |
| Fat Chef Biryani Wale |         6540.89 |
| Royal Treat           |         6476.97 |
| Mum?s Kitchen         |         6264.02 |
| The 13th Floor        |         5949.81 |
| Cafe D'hide           |         5817.08 |
| Five Star Chicken     |         5812.9  |
+-----------------------+-----------------+


In [17]:
#daily repeated order
mycursor.execute("""
SELECT order_date, COUNT(*) AS total_orders
FROM clean_ordersa
GROUP BY order_date
ORDER BY order_date
""")

out = mycursor.fetchall()

print(tabulate(
    out,
    headers=[i[0] for i in mycursor.description],
    tablefmt='psql'
))

+--------------+----------------+
| order_date   |   total_orders |
|--------------+----------------|
| 2025-04-29   |             13 |
| 2025-04-30   |             16 |
| 2025-05-01   |             10 |
| 2025-05-02   |             19 |
| 2025-05-03   |              9 |
| 2025-05-04   |             23 |
| 2025-05-05   |             12 |
| 2025-05-06   |             20 |
| 2025-05-07   |             20 |
| 2025-05-08   |             13 |
| 2025-05-09   |             12 |
| 2025-05-10   |             13 |
| 2025-05-11   |              9 |
| 2025-05-12   |             14 |
| 2025-05-13   |             15 |
| 2025-05-14   |             18 |
| 2025-05-15   |             14 |
| 2025-05-16   |             22 |
| 2025-05-17   |             16 |
| 2025-05-18   |             21 |
| 2025-05-19   |              7 |
| 2025-05-20   |             22 |
| 2025-05-21   |             15 |
| 2025-05-22   |             17 |
| 2025-05-23   |             12 |
| 2025-05-24   |             20 |
| 2025-05-25  

In [18]:
#which payement analysis people use often
mycursor.execute("""
SELECT payment_method, COUNT(*) AS usage_count
FROM clean_ordersa
GROUP BY payment_method
ORDER BY usage_count DESC
""")

out = mycursor.fetchall()

print(tabulate(
    out,
    headers=[i[0] for i in mycursor.description],
    tablefmt='psql'
))

+------------------+---------------+
| payment_method   |   usage_count |
|------------------+---------------|
| Cash             |          1552 |
| Upi              |          1510 |
| Card             |          1500 |
+------------------+---------------+
